# Getting started — LLM from scratch

This notebook checks that the environment is ready and sketches the path we will follow:

1. Tokenization (text → integer IDs)
2. Embeddings and positional encoding
3. Attention (scaled dot-product, then multi-head)
4. Transformer blocks (MLP, residual, layer norm)
5. Training loop and generation

**Kernel:** in Cursor, select **`.venv (Python 3.12)`**. That is the project virtual environment. A global/system Python kernel will fail with `No module named 'numpy'`.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

print("Interpreter:", sys.executable)

venv_python = Path.cwd().resolve()
if "notebooks" in venv_python.parts:
    venv_python = venv_python.parent
venv_python = venv_python / ".venv" / "Scripts" / "python.exe"

if Path(sys.executable).resolve() != venv_python.resolve():
    raise RuntimeError(
        "This notebook is not using the project virtual environment.\n"
        f"Current: {sys.executable}\n"
        f"Expected: {venv_python}\n"
        "In the kernel picker, select 'Python (LLM from scratch)' "
        "or the interpreter at .venv\\Scripts\\python.exe"
    )

import numpy as np
import pandas as pd
import torch
import tensorflow as tf

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("TensorFlow:", tf.__version__)
print("CUDA available (PyTorch):", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Interpreter: c:\Users\Arman Sayyed\Desktop\AI\LLM_from_scratch\.venv\Scripts\python.exe
Python: 3.12.12
NumPy: 2.5.2
Pandas: 3.0.5
PyTorch: 2.13.0+cpu
TensorFlow: 2.21.0
CUDA available (PyTorch): False


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

x = torch.randn(2, 4, device=device)
print("Sample tensor:\n", x)

Using device: cpu
Sample tensor:
 tensor([[ 0.2819, -0.3352,  0.3472,  0.5361],
        [-0.7070,  1.6718, -0.4823, -1.3576]])


## Tokenization preview

Language models train on token IDs, not raw strings. `tiktoken` is the BPE tokenizer used by GPT-style models.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
text = "Building an LLM from scratch."
ids = enc.encode(text)

print("Text:", text)
print("Token IDs:", ids)
print("Decoded:", enc.decode(ids))
print("Vocab size:", enc.n_vocab)

## Project layout

| Path | Role |
| --- | --- |
| `data/` | Datasets |
| `models/` | Model implementations |
| `src/` | Shared source code |
| `notebooks/` | Exploration (this file) |
| `tests/` | Unit tests |